In [1]:
import shutil
import os
import tqdm
import torch
import torch.nn as nn
import pandas as pd
import boda

/proj/bmfm/users/sanjoy/miniforge3/envs/malinois_python311/lib/python3.11/site-packages/lightning/fabric/__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.


# Load models interactively

This notebook describes how to load Malinois and use it for inference. It's important to remember that Malinois processes `(bsz, 4, 600)` tensors but was trained on 200-mers. Therefore you need to pad input sequences with 200 nucleotides on each side from the MPRA vector used to generate the data. This is done using the `FlankBuilder`.

# Get Malinois

Can download directly from a Google Storage bucket you can access.

In [2]:
malinois_path = 'gs://tewhey-public-data/CODA_resources/malinois_artifacts__20211113_021200__287348.tar.gz'
my_model = boda.common.utils.load_model(malinois_path)

Copying gs://tewhey-public-data/CODA_resources/malinois_artifacts__20211113_021200__287348.tar.gz...
/ [1 files][ 49.3 MiB/ 49.3 MiB]                                                
Operation completed over 1 objects/49.3 MiB.                                     
archive unpacked in ./


Loaded model from 20211113_021200 in eval mode


In [3]:
# Search all .vcf files inside the 'gs://tewhey-public-data' directory
vcf_files = boda.common.utils.search_files('gs://tewhey-public-data', '.vcf')

In [3]:
input_len = torch.load('./artifacts/torch_checkpoint.pt', weights_only=False)['model_hparams'].input_len
print(f'Input length: {input_len}')

Input length: 600


# Set flanks

MPRA flanks are saved as constants in the `boda` repo. These need to be sized to (1, 4, 200) each and used to init `FlankBuilder`.

In [4]:
left_pad_len = (input_len - 200) // 2
right_pad_len= (input_len - 200) - left_pad_len

left_flank = boda.common.utils.dna2tensor( 
    boda.common.constants.MPRA_UPSTREAM[-left_pad_len:] 
).unsqueeze(0)
print(f'left flank shape: {left_flank.shape}')

right_flank= boda.common.utils.dna2tensor( 
    boda.common.constants.MPRA_DOWNSTREAM[:right_pad_len] 
).unsqueeze(0)
right_flank.shape
print(f'right flank shape: {right_flank.shape}')

flank_builder = boda.common.utils.FlankBuilder(
    left_flank=left_flank,
    right_flank=right_flank,
)

flank_builder.cuda()

left flank shape: torch.Size([1, 4, 200])
right flank shape: torch.Size([1, 4, 200])


FlankBuilder()

# Example call

Using `torch.no_grad()` so the computation graph isn't saved to memory. Since sequences are passed to the model as onehots in `torch.float32` format, we can use `torch.randn` to validate the model setup. Here a batch of 10 variable 200 nt (fake) sequences are being padded to 600 nt, then being passed to the model. Note, `my_model` and `flank_builder` have been set on the GPU using `.cuda()` calls. Therefore, the fake sequence also needs to be sent to `cuda`.

Note: this fake sequence will result in pathological predictions, it's only an illustrative example.

In [5]:
placeholder = torch.randn((10,4,200)).cuda() # Simulate a batch_size x 4 nucleotide x 200 nt long sequence
prepped_seq = flank_builder( placeholder )   # Need to add MPRA flanks

with torch.no_grad():
    print( my_model( prepped_seq ) )


In [6]:
prepped_seq.shape

# Run on the TeWhey datasets from Siraj et al. 

that I curated from Supp Table 4 with emVar=True... Note that there are a few thousands sample more in the Butts et al. in all cell lines... Also, I curated the dataset using a script on process_snv_datasets of CCC /dccstor/bmfm-targets1/users/sanjoy/tools/boda2 using bmfm env...

In [20]:
import pandas as pd
import numpy as np
import csv
from scipy.stats import pearsonr, spearmanr
import tqdm.notebook as tqdm
import matplotlib.pyplot as plt

In [8]:
input_filename = "/proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/Siraj_data/skew_snv_mpra_Tewhey_emVars.csv"

mpra_skewness_data = pd.read_table(input_filename, sep=',', header=0)
#print(mpra_skewness_data.columns, mpra_skewness_data.shape, mpra_skewness_data.head())
cell_lines = ['A549', 'K562', 'HEPG2', 'SKNSH', 'HCT116']
for cell in cell_lines:
    # Cast 'emVar_K562' to boolean and 'log2Skew_K562' to float and 'log2FC_K562' to float
    mpra_skewness_data[f"emVar_{cell}"] = mpra_skewness_data[f"emVar_{cell}"].astype(float)
    mpra_skewness_data[f"allele1.log2FC_{cell}"] = mpra_skewness_data[f"allele1.log2FC_{cell}"].astype(float)
    mpra_skewness_data[f"allele2.log2FC_{cell}"] = mpra_skewness_data[f"allele2.log2FC_{cell}"].astype(float)
    mpra_skewness_data[f"log2Skew_{cell}"] = mpra_skewness_data[f"log2Skew_{cell}"].astype(float)
    mpra_skewness_data[f"log2Skew.SE_{cell}"] = mpra_skewness_data[f"log2Skew.SE_{cell}"].astype(float)
    mpra_skewness_data[f"log2Skew.FDR_{cell}"] = mpra_skewness_data[f"log2Skew.FDR_{cell}"].astype(float)
    
    print(mpra_skewness_data[f'emVar_{cell}'].value_counts())
print(mpra_skewness_data.columns, mpra_skewness_data.shape, mpra_skewness_data.head())
#mpra_19 = mpra_19.loc[ mpra_19.loc[:, ['K562_lfcSE', 'HepG2_lfcSE', 'SKNSH_lfcSE']].max(axis=1) < 1.0 ]


In [13]:
# Build a dataset by taking ("seqR", allele1. and "seqA" as rows and keeping the 

df = mpra_skewness_data.copy()
print("Shape of original dataframe:", df.shape)
allele1_cols = [c for c in df.columns if c.startswith("allele1.log2FC_")]
allele2_cols = [c for c in df.columns if c.startswith("allele2.log2FC_")]

# Keep original index
df_allele1 = df[['variant', 'seqR'] + allele1_cols].copy()
df_allele1 = df_allele1.rename(columns={'seqR': 'sequence'})
df_allele1 = df_allele1.rename(columns=lambda x: x.replace('allele1.log2FC_', 'log2FC_'))
df_allele1['allele'] = 'allele1'

df_allele2 = df[['variant', 'seqA'] + allele2_cols].copy()
df_allele2 = df_allele2.rename(columns={'seqA': 'sequence'})
df_allele2 = df_allele2.rename(columns=lambda x: x.replace('allele2.log2FC_', 'log2FC_'))
df_allele2['allele'] = 'allele2'

# Add original row index
df_allele1['_idx'] = df.index
df_allele2['_idx'] = df.index

# Concatenate and sort to interleave
df_long = (
    pd.concat([df_allele1, df_allele2])
      .sort_values(['_idx', 'allele'])
      .drop(columns='_idx')
      .reset_index(drop=True)
)

df_long['sequence'].tail(10)
# Remove the rows where 'sequence' contains 'NaN'
df_long = df_long[ ~df_long['sequence'].isna() ].reset_index(drop=True)
print(df_long.columns, df_long.shape, df_long.head())
# if the length of the sequence is not 201, then take the first 201 characters of the sequence
df_long['sequence'] = df_long['sequence'].str.slice(0, 201)
print(df_long.columns, df_long.shape, df_long.head(10))


In [14]:
# Now get the prediction for all seqs at a glance

seq_tensor  = torch.stack([ boda.common.utils.dna2tensor(x['sequence']) for i, x in tqdm.tqdm(df_long.iterrows(), total=df_long.shape[0]) ], dim=0)
seq_dataset = torch.utils.data.TensorDataset(seq_tensor)
seq_loader  = torch.utils.data.DataLoader(seq_dataset, batch_size=128)

In [15]:
results = []

with torch.no_grad():
    for i, batch in enumerate(tqdm.tqdm(seq_loader)):
        prepped_seq = flank_builder( batch[0].cuda() )
        predictions = my_model( prepped_seq ) + \
                      my_model( prepped_seq.flip(dims=[1,2]) ) # Also
        predictions = predictions.div(2.)
        results.append(predictions.detach().cpu())
                
predictions = torch.cat(results, dim=0) 

In [16]:
predictions.shape

In [21]:
pred_df     = pd.DataFrame( predictions.numpy(), columns=['K562_preds', 'HEPG2_preds', 'SKNSH_preds'] )
all_results = pd.concat([df_long, pred_df], axis=1)
all_results

In [26]:
# Get the Pearsons R for Log2FC vs Preds for each cell line
for cell in ['K562', 'HEPG2', 'SKNSH']:
    df = all_results[[f'log2FC_{cell}', f'{cell}_preds']].dropna()
    corr = pearsonr(df[f'log2FC_{cell}'], df[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')
    
    corr = all_results[f'log2FC_{cell}'].corr(all_results[f'{cell}_preds'])
    print(cell, f'{corr:.4f}')

# Now look for the skewness prediction as the diff (delta)

In [32]:
cols = [
    'log2FC_A549',
    'log2FC_K562',
    'log2FC_HEPG2',
    'log2FC_SKNSH',
    'log2FC_HCT116',
    'K562_preds',
    'HEPG2_preds',
    'SKNSH_preds'
]


skew_df = (
    all_results
    .pivot(index='variant', columns='allele', values=cols)
)

skew_df = (
    skew_df.xs('allele1', level=1, axis=1)
    - skew_df.xs('allele2', level=1, axis=1)
)

skew_df = skew_df.add_prefix('skewness_')
print(skew_df.columns, skew_df.shape, skew_df.head())


In [ ]:
# Compute the correlation between the skewness of log2FC and the skewness of predictions for each cell line
for cell in [ 'K562', 'HEPG2', 'SKNSH']:
    df = skew_df[[f'skewness_log2FC_{cell}', f'skewness_{cell}_preds']].dropna()
    corr = pearsonr(df[f'skewness_log2FC_{cell}'], df[f'skewness_{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')
    
    corr = skew_df[f'skewness_log2FC_{cell}'].corr(skew_df[f'skewness_{cell}_preds'])
    print(cell, f'{corr:.4f}')

In [ ]:
# Plot the correlation between the skewness of log2FC and the skewness of predictions for each cell line
for cell in [ 'K562', 'HEPG2', 'SKNSH']:    
    plt.figure(figsize=(6,6))
    plt.scatter(skew_df[f'skewness_log2FC_{cell}'], skew_df[f'skewness_{cell}_preds'], alpha=0.5)
    plt.xlabel(f'Skewness of log2FC_{cell}')
    plt.ylabel(f'Skewness of {cell}_preds')
    # Put the scale of x-axis and y-axis to be the same [-6 6]
    plt.xlim([-6, 6])
    plt.ylim([-6, 6])
    plt.title(f'Correlation between skewness of log2FC and predictions for {cell}')
    # Put the correlation coefficient as the legend in the top left corner
    corr = skew_df[f'skewness_log2FC_{cell}'].corr(skew_df[f'skewness_{cell}_preds'])
    plt.legend([f'Pearson R: {corr:.4f}'], loc='upper left')
    plt.grid()
    plt.show()  
    

# Now load the Original Aggarwal data and see the score... 


In [ ]:
cell_lines = [ 'K562', 'HEPG2']
cell_lines = 'K562'
# This is for the original whole data reporting...
# input_filename = f"/proj/bmfm/datasets/omics/genome/finetune_datasets/lenti_mpra_regression/{cell_lines}_clean.csv"
# df_long = pd.read_csv(input_filename, sep = '\t', header=0)

# This is for the test reporting... 
input_filename = f"/proj/bmfm/datasets/omics/genome/finetune_datasets/lenti_mpra_regression/{cell_lines}_original_trimmed/test.csv"

df_long = pd.read_csv(input_filename, header=0)
print(df_long.columns, df_long.shape, df_long.head())



Index(['chunk', 'mean_value'], dtype='str') (43588, 2)                                                chunk  mean_value
0  CATCTACATAGAAGTCGCCCTGTCCGTGATGTCACCGACAGTGCCT...       0.628
1  TTGCTCCTTAACACAGGCTAAGGACCAGCTTCTTTGGGAGAGAACA...       1.263
2  TCCCTGGTGGTCTAGTGGTTAGGATTCGGCGCTCTCACCGCCGCGG...      -0.924
3  CAGGTAACTACTCTGCAAAATGAGGACACCAGGTGCGTTTTGCCCT...      -0.613
4  TTTTGTATTTTTAGTAGAGACGGGGTTTCTCCATGTTGGTCATGCT...      -0.542


In [16]:
# Now get the prediction for all seqs at a glance
seq_col_name = 'chunk' # Change this to the actual column name in the dataframe that contains the sequences
seq_tensor  = torch.stack([ boda.common.utils.dna2tensor(x[seq_col_name]) for i, x in tqdm.tqdm(df_long.iterrows(), total=df_long.shape[0]) ], dim=0)
seq_dataset = torch.utils.data.TensorDataset(seq_tensor)
seq_loader  = torch.utils.data.DataLoader(seq_dataset, batch_size=128)


100%|██████████████████████████████████████████████████| 43588/43588 [00:03<00:00, 12564.28it/s]


In [17]:
results = []

with torch.no_grad():
    for i, batch in enumerate(tqdm.tqdm(seq_loader)):
        prepped_seq = flank_builder( batch[0].cuda() )
        predictions = my_model( prepped_seq ) + \
                      my_model( prepped_seq.flip(dims=[1,2]) ) # Also
        predictions = predictions.div(2.)
        results.append(predictions.detach().cpu())
                
predictions = torch.cat(results, dim=0) 
predictions.shape

100%|█████████████████████████████████████████████████████████| 341/341 [00:07<00:00, 43.16it/s]


torch.Size([43588, 3])

In [18]:
pred_df     = pd.DataFrame( predictions.numpy(), columns=['K562_preds', 'HEPG2_preds', 'SKNSH_preds'] )
all_results = pd.concat([df_long, pred_df], axis=1)
all_results.shape, all_results.columns, all_results.head()

((43588, 5),
 Index(['chunk', 'mean_value', 'K562_preds', 'HEPG2_preds', 'SKNSH_preds'], dtype='str'),
                                                chunk  mean_value  K562_preds  \
 0  CATCTACATAGAAGTCGCCCTGTCCGTGATGTCACCGACAGTGCCT...       0.628    4.985993   
 1  TTGCTCCTTAACACAGGCTAAGGACCAGCTTCTTTGGGAGAGAACA...       1.263    5.114730   
 2  TCCCTGGTGGTCTAGTGGTTAGGATTCGGCGCTCTCACCGCCGCGG...      -0.924    1.570006   
 3  CAGGTAACTACTCTGCAAAATGAGGACACCAGGTGCGTTTTGCCCT...      -0.613    2.034996   
 4  TTTTGTATTTTTAGTAGAGACGGGGTTTCTCCATGTTGGTCATGCT...      -0.542    0.462392   
 
    HEPG2_preds  SKNSH_preds  
 0     4.784856     7.241030  
 1     4.951921     7.386204  
 2     0.948694     1.033938  
 3     1.387128     1.062202  
 4     0.449989     0.463651  )

In [21]:
print(cell_lines.upper())
df = all_results[['mean_value', f'{cell_lines.upper()}_preds']].dropna()
corr = pearsonr(df['mean_value'], df[f'{cell_lines.upper()}_preds'])
print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

results = []

for fold in sorted(all_results['fold'].unique()):
    
    df_fold = (
        all_results[all_results['fold'] == fold]
        [['mean_value', f'{cell_lines.upper()}_preds']]
        .dropna()
    )
    
    # Skip folds with too few samples
    if len(df_fold) < 2:
        continue

    r, p = pearsonr(df_fold['mean_value'], df_fold[f'{cell_lines.upper()}_preds'])

    results.append(r)
    
    print(f"Fold {fold}: r = {r:.4f}, p = {p:.4e}")

# Average correlation across folds
avg_r = np.mean(results)

print("\nAverage Pearson r across folds:", round(avg_r, 4))


K562
stat: 0.6926, pvalue: 0.0


  0%|                                                                 | 0/43588 [02:36<?, ?it/s]


KeyError: 'fold'

In [ ]:
# Get the Pearsons R for Log2FC vs Preds for each cell line
for cell in ['K562', 'HEPG2', 'SKNSH']:
    df = all_results[[f'log2FC_{cell}', f'{cell}_preds']].dropna()
    corr = pearsonr(df[f'log2FC_{cell}'], df[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')
    
    corr = all_results[f'log2FC_{cell}'].corr(all_results[f'{cell}_preds'])
    print(cell, f'{corr:.4f}')

# First attempt on K562 only...

In [26]:
input_filename = "/proj/bmfm/datasets/omics/genome/finetune_datasets/snv_mpra_Tewhey/Siraj_data/skew_snv_mpra_Tewhey_K562_emVars_refalt_separate.csv"

mpra_skewness_data = pd.read_table(input_filename, sep=',', header=0)
# Cast 'emVar_K562' to boolean and 'log2Skew_K562' to float and 'log2FC_K562' to float
mpra_skewness_data['emVar_K562'] = mpra_skewness_data['emVar_K562'].astype(bool)
mpra_skewness_data['log2Skew_K562'] = mpra_skewness_data['log2Skew_K562'].astype(float)
mpra_skewness_data['log2FC_K562'] = mpra_skewness_data['log2FC_K562'].astype(float)
print(mpra_skewness_data.columns, mpra_skewness_data.shape, mpra_skewness_data.head())
print(mpra_skewness_data['emVar_K562'].value_counts())
#mpra_19 = mpra_19.loc[ mpra_19.loc[:, ['K562_lfcSE', 'HepG2_lfcSE', 'SKNSH_lfcSE']].max(axis=1) < 1.0 ]


In [21]:
39258/2

In [10]:
pass_seq = mpra_skewness_data.loc[ mpra_skewness_data['sequence'].str.len() == 201 ].reset_index(drop=True)
print(pass_seq.columns, pass_seq.shape, pass_seq.head())

# Find the sequences that have log2FC_K562 values that are not NaN and log2Skew_K562 greater than 0.5 or less than -0.5
pass_seq = pass_seq.loc[
    pass_seq['log2FC_K562'].notna() & 
    pass_seq['emVar_K562'] &
    (
        (pass_seq['log2Skew_K562'] > 0.5) |
        (pass_seq['log2Skew_K562'] < -0.5)
    )
].reset_index(drop=True)
print(pass_seq.columns, pass_seq.shape, pass_seq.head())

#plot the distribution of log2FC_K562 values...
plt.hist(pass_seq['log2FC_K562'], bins=50)
plt.xlabel('log2FC_K562')
plt.ylabel('Frequency')     
plt.show()

# seq_tensor  = torch.stack([ boda.common.utils.dna2tensor(x['sequence']) for i, x in tqdm.tqdm(pass_seq.iterrows(), total=pass_seq.shape[0]) ], dim=0)
# seq_dataset = torch.utils.data.TensorDataset(seq_tensor)
# seq_loader  = torch.utils.data.DataLoader(seq_dataset, batch_size=128)

In [11]:

seq_tensor  = torch.stack([ boda.common.utils.dna2tensor(x['sequence']) for i, x in tqdm.tqdm(pass_seq.iterrows(), total=pass_seq.shape[0]) ], dim=0)
seq_dataset = torch.utils.data.TensorDataset(seq_tensor)
seq_loader  = torch.utils.data.DataLoader(seq_dataset, batch_size=128)

In [12]:
results = []

with torch.no_grad():
    for i, batch in enumerate(tqdm.tqdm(seq_loader)):
        prepped_seq = flank_builder( batch[0].cuda() )
        predictions = my_model( prepped_seq ) + \
                      my_model( prepped_seq.flip(dims=[1,2]) ) # Also
        predictions = predictions.div(2.)
        results.append(predictions.detach().cpu())
                
predictions = torch.cat(results, dim=0) 

In [13]:
predictions.shape

In [14]:
pred_df     = pd.DataFrame( predictions.numpy(), columns=['K562_preds', 'HepG2_preds', 'SKNSH_preds'] )
all_results = pd.concat([pass_seq, pred_df], axis=1)
all_results

In [17]:
# create a col 'chr' by splitting 'variant' on ':' and taking the first part and remove the 'chr' prefix if it exists, and create a col 'pos' by splitting 'variant' on ':' and taking the second part
all_results['chr'] = all_results['variant'].str.split(':').str[0].str.replace('chr', '')
all_results['pos'] = all_results['variant'].str.split(':').str[1]
all_results

In [19]:
chr_filter = (all_results['chr'] == 19) | \
             (all_results['chr'] == 21) | \
             (all_results['chr'] == '19') | \
             (all_results['chr'] == '21') | \
             (all_results['chr'] == 'X')

val_results = all_results.loc[ chr_filter ]
print(val_results.columns, val_results.shape, val_results.head())

In [20]:
for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = pearsonr(val_results[f'log2FC_{cell}'], val_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

# Run on MPRA data set

We're focusing on sequences that are 200 nt long for simplicity. In the paper we padded smaller sequences with additional nucleotides from the flanks.

In [34]:
!gsutil cp gs://tewhey-public-data/CODA_resources/Table_S2__MPRA_dataset.txt ./
mpra_19 = pd.read_table('Table_S2__MPRA_dataset.txt', sep='\t', header=0)

mpra_19 = mpra_19.loc[ mpra_19.loc[:, ['K562_lfcSE', 'HepG2_lfcSE', 'SKNSH_lfcSE']].max(axis=1) < 1.0 ]

In [35]:
pass_seq = mpra_19.loc[ mpra_19['sequence'].str.len() == 200 ].reset_index(drop=True)

seq_tensor  = torch.stack([ boda.common.utils.dna2tensor(x['sequence']) for i, x in tqdm.tqdm(pass_seq.iterrows(), total=pass_seq.shape[0]) ], dim=0)
seq_dataset = torch.utils.data.TensorDataset(seq_tensor)
seq_loader  = torch.utils.data.DataLoader(seq_dataset, batch_size=128)

In [44]:
print(pass_seq.shape)

print(seq_tensor[0].shape)

In [45]:
results = []

with torch.no_grad():
    for i, batch in enumerate(tqdm.tqdm(seq_loader)):
        prepped_seq = flank_builder( batch[0].cuda() )
        predictions = my_model( prepped_seq ) + \
                      my_model( prepped_seq.flip(dims=[1,2]) ) # Also
        predictions = predictions.div(2.)
        results.append(predictions.detach().cpu())
                
predictions = torch.cat(results, dim=0)

In [51]:
predictions.shape

In [12]:
pred_df     = pd.DataFrame( predictions.numpy(), columns=['K562_preds', 'HepG2_preds', 'SKNSH_preds'] )
all_results = pd.concat([pass_seq, pred_df], axis=1)
all_results

# Validation set performance
Check performance on chromosomes 19, 21, and X (held-out for validation during hparam selection).

In [13]:
chr_filter = (all_results['chr'] == 19) | \
             (all_results['chr'] == 21) | \
             (all_results['chr'] == '19') | \
             (all_results['chr'] == '21') | \
             (all_results['chr'] == 'X')

val_results = all_results.loc[ chr_filter ]

## Pearson's r

In [14]:
for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = pearsonr(val_results[f'{cell}_log2FC'], val_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

## Spearman's rho

In [15]:
for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = spearmanr(val_results[f'{cell}_log2FC'], val_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

# Test set performance
Check performance on chromosomes 7 and 13 (held-out for final testing, not used for model selection).

In [16]:
chr_filter = (all_results['chr'] == 7) | \
             (all_results['chr'] == 13) | \
             (all_results['chr'] == '7') | \
             (all_results['chr'] == '13')

test_results = all_results.loc[ chr_filter ]

## Pearson's r

In [17]:
for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = pearsonr(test_results[f'{cell}_log2FC'], test_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')

## Spearman's rho

In [18]:
test_results = all_results.loc[ chr_filter ]

for cell in ['K562', 'HepG2', 'SKNSH']:
    corr = spearmanr(test_results[f'{cell}_log2FC'], test_results[f'{cell}_preds'])
    print(cell)
    print(f'stat: {corr[0]:.4f}, pvalue: {corr[1]}')